# Interpretability on Synthetic Datasets

The goal is to show **when SurvLIME and SurvSHAP(t) work well and where their limitations emerge**, using two controlled synthetic datasets with known ground-truth feature effects.

| Dataset | Effects | Goal |
| ------- | ------- | ---- |
| A       | constant | Sanity-check for SurvLIME (Cox-like surrogate) |
| B       | time-varying on x1 ($\beta_1(t) = -1 + 2t$) | Stress-test for SurvSHAP(t) (does it capture dynamics?) |

Black-box models: **CoxPH** (glass-box / baseline) and **RSF** (flexible black-box). RSF matters because post-hoc explanation methods are most needed for such models.

The DGM reproduces the design from MI2DataLab/survshap (Krzyzinski et al., KBS 2023; arXiv:2208.11080).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.data import generate_interpretability_data, generate_tv_interpretability_data
from src.models import ALL_MODELS
from src.evaluation.metrics import concordance_index_ipcw, integrated_brier_score
from src.interpretability import SurvLIMEExplainer, SurvSHAPExplainer

sns.set_theme(style='whitegrid', palette='colorblind')
SEED = 42
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

IMPORTANT_FEATURES = ['x0', 'x1', 'x2', 'x3']
TRUE_SIGNS = {'x0': 1.0, 'x1': 1.0, 'x2': -1.0, 'x3': 1.0}
TRUE_BETAS_A = np.array([1.0, 0.5, -0.2, 0.1, 1e-6, 1e-6, 1e-6, 1e-6])


## 1. Data

The DGM reproduces the design from MI2DataLab/survshap (`paper/other_codes/data_generation.R`). Shared feature distribution:

| Feature   | Distribution     | Role     |
| --------- | ---------------- | -------- |
| x0, x1    | Bernoulli(0.5)   | binary, important |
| x2        | $\mathcal N(10, 2)$ | continuous, important |
| x3        | $\mathcal N(20, 4)$ | continuous, important |
| x4..x7    | $\mathcal N(0, 1)$  | noise (true $\beta = 10^{-6}$) |

Baseline hazard: $h_0 = 0.08$ (paper "exponential" variant). Censoring: administrative $C_a \sim U(11, 16)$ and random $C_r \sim U(0, 24)$, $T_{obs} = \min(T, C_a, C_r)$. The Brent sampler time grid is bounded to $t \in [10^{-8}, 20]$ (paper convention) — this hard cap prevents $H(t)$ blowup.

### Dataset A — constant betas

$$\beta = (1.0,\ 0.5,\ -0.2,\ 0.1,\ 10^{-6},\ 10^{-6},\ 10^{-6},\ 10^{-6})$$

Coefficients are chosen so that $|\beta \cdot x|$ is O(1) for each informative feature despite different feature scales.

### Dataset B — time-varying $\beta_1$

$$\beta_0 = 1.0,\quad \beta_1(t) = -1 + 2t,\quad \beta_2 = -0.2,\quad \beta_3 = 0.1$$

TV only on $x_1$ (Bernoulli): sign change at $t = 0.5$, contribution $\in [-1, 1]$. Making $\beta_3(t)$ time-varying does not work — on $x_3 \sim \mathcal N(20, 4)$ the negative slope dominates, $B(x) = 2x_1 - 0.2 x_3$ goes negative for most subjects → cure fraction → 95%+ censoring. One TV feature is enough to showcase SurvSHAP(t)'s time dimension.

Expected: $x_1$ is TV; $x_0$, $x_2$, $x_3$ are consistently important; noise ≈ 0.


In [ ]:
# Dataset A: PH, constant effects
data_interp = generate_interpretability_data(seed=SEED)
print('=== Dataset A (PH, constant effects) ===')
print(data_interp.summary())
print(f'True betas: {data_interp.true_betas}')

X_int_tr, X_int_te, T_int_tr, T_int_te, E_int_tr, E_int_te = train_test_split(
    data_interp.X, data_interp.T, data_interp.E,
    test_size=0.2, random_state=SEED, stratify=data_interp.E.astype(int),
)
feature_names = data_interp.feature_names

event_times_int = T_int_te[E_int_te.astype(bool)]
interp_times = np.linspace(event_times_int.min() + 1e-6, np.percentile(event_times_int, 95), 50)


In [ ]:
# Dataset B: Non-PH, time-varying x1
data_tv = generate_tv_interpretability_data(seed=SEED)
print('=== Dataset B (Non-PH, time-varying x1) ===')
print(data_tv.summary())
print('beta1(t) = -1 + 2t   (sign change at t = 0.5);  beta3 = 0.1 (constant)')

X_tv_tr, X_tv_te, T_tv_tr, T_tv_te, E_tv_tr, E_tv_te = train_test_split(
    data_tv.X, data_tv.T, data_tv.E,
    test_size=0.2, random_state=SEED, stratify=data_tv.E.astype(int),
)

event_times_tv = T_tv_te[E_tv_te.astype(bool)]
tv_times = np.linspace(event_times_tv.min() + 1e-6, np.percentile(event_times_tv, 95), 50)


## 2. Models

Dataset A uses three models: **CoxPH** (parametric, glass-box), **DeepSurv** (neural Cox, intermediate), **RSF** (non-parametric black-box). Dataset B uses only CoxPH + RSF — DeepSurv adds nothing fundamentally new to the TV-effect question.

Before trusting the explanations — sanity check on black-box quality (IPCW C-index + IBS).


In [ ]:
# Fit CoxPH + DeepSurv + RSF on Dataset A; CoxPH + RSF on Dataset B
MODELS_A = ['cox_ph', 'deep_surv', 'rsf']

interp_models = {}
for mname in MODELS_A:
    kwargs = {'hidden_layers': [32]} if mname == 'deep_surv' else {}
    m = ALL_MODELS[mname](**kwargs)
    m.fit(X_int_tr, T_int_tr, E_int_tr)
    interp_models[mname] = m
    print(f'Dataset A: {mname} fitted.')

tv_models = {}
for mname in ['cox_ph', 'rsf']:
    m = ALL_MODELS[mname]()
    m.fit(X_tv_tr, T_tv_tr, E_tv_tr)
    tv_models[mname] = m
    print(f'Dataset B: {mname} fitted.')


In [ ]:
# Black-box sanity check
sanity_rows = []
for tag, mods, X_tr, T_tr, E_tr, X_te, T_te, E_te, times in [
    ('A (PH)',     interp_models, X_int_tr, T_int_tr, E_int_tr, X_int_te, T_int_te, E_int_te, interp_times),
    ('B (Non-PH)', tv_models,     X_tv_tr,  T_tv_tr,  E_tv_tr,  X_tv_te,  T_tv_te,  E_tv_te,  tv_times),
]:
    for mname, m in mods.items():
        risk = m.predict_risk(X_te)
        c_ipcw = concordance_index_ipcw(T_tr, E_tr, T_te, E_te, risk)
        try:
            sf = m.predict_survival_function(X_te, times)
            ibs = integrated_brier_score(T_tr, E_tr, T_te, E_te, sf, times)
        except Exception:
            ibs = float('nan')
        sanity_rows.append({'dataset': tag, 'model': mname,
                            'c_index_ipcw': c_ipcw, 'ibs': ibs})

sanity_df = pd.DataFrame(sanity_rows)
print('=== Synthetic black-box sanity check ===')
print(sanity_df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))


In [ ]:
# CoxPH global betas — reference for SurvLIME comparison.
# These are the population-level Cox coefficients fit on the full training set.
# SurvLIME approximates a *local* Cox surrogate around each individual; on a
# Cox-PH-generated dataset the local coefficients should agree with the global
# CoxPH fit (Dataset A), and disagree on the time-varying dataset (Dataset B,
# where the constant Cox fit averages beta1(t) over the observation window).

# Dataset A — true betas are constant
beta_cox_a = interp_models['cox_ph']._model.coef_
TV_BETAS_B_T0 = np.array([1.0, -1.0, -0.2, 0.1, 1e-6, 1e-6, 1e-6, 1e-6])  # beta1(t=0) = -1
beta_cox_b = tv_models['cox_ph']._model.coef_

cox_betas = pd.DataFrame({
    'feature':       feature_names,
    'true_A':        TRUE_BETAS_A,
    'CoxPH_A':       beta_cox_a,
    'true_B (t=0)':  TV_BETAS_B_T0,
    'CoxPH_B':       beta_cox_b,
})

print('=== CoxPH global betas vs ground truth ===')
print('Dataset A: betas are constant — CoxPH should recover them closely.')
print('Dataset B: beta1(t) = -1 + 2t — CoxPH fits a single coefficient ≈ time-average of beta1(t)')
print('           over the observation window.  Other betas should be close to truth.')
print()
print(cox_betas.to_string(index=False, float_format=lambda v: f'{v:+.3f}'))
cox_betas


## 3. Evaluation on Dataset A

Primary sanity check for SurvLIME: its surrogate model is Cox-like, so we can directly compare SurvLIME coefficients with the true $\beta$.


### 3.1 SurvLIME

Local Cox surrogate around each individual → one coefficient per feature. Three models:

| Model     | Role                                            |
| --------- | ----------------------------------------------- |
| CoxPH     | glass-box; SurvLIME surrogate matches the DGP   |
| DeepSurv  | neural Cox; same structure but nonlinear         |
| RSF       | non-parametric; surrogate is the least similar  |

Expected: CoxPH closest to true β; DeepSurv intermediate; RSF noisier, but signs and ranking should hold.


In [ ]:
# Select 3 individuals by RSF risk percentile (P10, P50, P90)
risk_scores = interp_models['rsf'].predict_risk(X_int_te)
percentiles = [10, 50, 90]
individual_indices = []
individual_labels = []

for p in percentiles:
    threshold = np.percentile(risk_scores, p)
    diffs = np.abs(risk_scores - threshold)
    diffs[individual_indices] = np.inf
    idx = int(np.argmin(diffs))
    individual_indices.append(idx)
    individual_labels.append(f'P{p} (risk={risk_scores[idx]:.3f})')

print('Selected individuals:')
for idx, label in zip(individual_indices, individual_labels):
    print(f'  [{idx}] {label}')


In [ ]:
# SurvLIME — CoxPH / DeepSurv / RSF side by side (P10 / P50 / P90, num_samples=500)
from src.interpretability import SurvLIMEExplainer

fig, axes = plt.subplots(3, 3, figsize=(18, 15))

lime_coefs = {}
for row, mname in enumerate(MODELS_A):
    lime_expl = SurvLIMEExplainer(
        interp_models[mname], X_int_tr, T_int_tr, E_int_tr,
    )
    lime_coefs[mname] = {}

    for col, (idx, label) in enumerate(zip(individual_indices, individual_labels)):
        coefs = lime_expl.explain(X_int_te[idx], num_samples=500)
        lime_coefs[mname][idx] = coefs
        lime_expl.plot_weights(coefs, feature_names, true_important=IMPORTANT_FEATURES, ax=axes[row, col])
        axes[row, col].set_title(f'{mname} — {label}', fontsize=10)

fig.suptitle('SurvLIME Local Feature Importance (Dataset A)', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()


In [ ]:
# SurvLIME quantitative recovery — Dataset A (Sign Recovery, Rank Agreement, Precision@k)
from scipy.stats import spearmanr

N_LIME = 10
rng_lime = np.random.default_rng(SEED)
lime_expl_idx = rng_lime.choice(len(X_int_te), size=N_LIME, replace=False)

imp_idx = [feature_names.index(n) for n in IMPORTANT_FEATURES]
k_imp   = len(imp_idx)
beta_S  = TRUE_BETAS_A[imp_idx]

lime_summary = []
for mname in MODELS_A:
    lime_expl = SurvLIMEExplainer(interp_models[mname], X_int_tr, T_int_tr, E_int_tr)
    all_coefs = np.stack([
        lime_expl.explain(X_int_te[i], num_samples=200) for i in lime_expl_idx
    ])  # (N_LIME, p)

    sign_rec_per = []
    rank_agr_per = []
    prec_k_per   = []

    for coefs in all_coefs:
        b_hat_S = coefs[imp_idx]

        sign_rec_per.append(float(np.mean(np.sign(b_hat_S) == np.sign(beta_S))))

        rho, _ = spearmanr(np.abs(beta_S), np.abs(b_hat_S))
        if not np.isnan(rho):
            rank_agr_per.append(rho)

        top_k_set = set(np.argsort(np.abs(coefs))[::-1][:k_imp])
        prec_k_per.append(len(top_k_set & set(imp_idx)) / k_imp)

    lime_summary.append({
        'model':          mname,
        'N':              N_LIME,
        'sign_recovery':  f'{np.mean(sign_rec_per):.3f}',
        'rank_agreement': f'{np.mean(rank_agr_per):.3f}',
        'precision@k':    f'{np.mean(prec_k_per):.3f}',
    })
    for ki, i in enumerate(lime_expl_idx):
        lime_coefs.setdefault(mname, {})[int(i)] = all_coefs[ki]

lime_recovery_df = pd.DataFrame(lime_summary)
print(f'=== SurvLIME quantitative recovery (Dataset A, N={N_LIME}, k={k_imp}) ===')
print(f'True betas (informative): {beta_S}')
print()
print(lime_recovery_df.to_string(index=False))
lime_recovery_df

### 3.2 SurvSHAP(t)

SurvSHAP(t) explains the survival function (target: cumulative hazard $H(t)$). For quantitative comparison with true $\beta$: $\hat{b}_i = \frac{1}{|\mathcal{T}|}\sum_{t \in \mathcal{T}} \phi_i(t)$ — mean over the evaluation time grid.

Same metrics: Sign Recovery, Rank Agreement, Precision@k.

Expected on Dataset A: SurvSHAP(t) curves for x0–x3 are nearly flat and monotone (constant effects); time-averaged $\hat{b}_i$ are close to true $\beta_i$ → high Sign Recovery and Rank Agreement.

In [ ]:
# SurvSHAP(t) — Dataset A: CoxPH / DeepSurv / RSF (P50 individual)
risk_int = interp_models['rsf'].predict_risk(X_int_te)
idx_a = int(np.argmin(np.abs(risk_int - np.percentile(risk_int, 50))))
print(f'Dataset A — selected individual: index={idx_a}, risk={risk_int[idx_a]:.3f} (P50)')

shap_results_a = {}
for mname in MODELS_A:
    expl = SurvSHAPExplainer(
        interp_models[mname], X_int_tr, T_int_tr, E_int_tr,
        feature_names=feature_names,
        function_type="chf",
    )
    shap_results_a[mname] = expl.explain(X_int_te[idx_a], interp_times)
    print(f'  {mname}: done')

_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
feat_color = {name: _colors[i % len(_colors)] for i, name in enumerate(sorted(feature_names))}

fig, axes = plt.subplots(2, 3, figsize=(22, 10), sharex=True)
fig.suptitle(
    'SurvSHAP(t) — Dataset A (Cox PH, constant effects) — P50 individual\n'
    'Target: cumulative hazard H(t). Expected: flat, monotone lines.',
    fontsize=12,
)

for col, mname in enumerate(MODELS_A):
    result_df = shap_results_a[mname]
    times_a, var_names_a, shap_mat = SurvSHAPExplainer._extract_shap_matrix(result_df)
    norm_mat = SurvSHAPExplainer._normalize_shap(shap_mat)

    for row, (matrix, ylabel) in enumerate(zip(
        [shap_mat, norm_mat],
        ['SurvSHAP(t) for H(t)', 'Normalized SurvSHAP(t)'],
    )):
        ax = axes[row, col]
        for i, name in enumerate(var_names_a):
            if name in IMPORTANT_FEATURES:
                ax.plot(times_a, matrix[i], label=name, linewidth=2, color=feat_color[name])
            else:
                ax.plot(times_a, matrix[i], alpha=0.2, color='gray', linewidth=0.8)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(mname.upper().replace('_', ' '))
        if row == 1:
            ax.set_xlabel('Time')
        if col == 2 and row == 0:
            ax.legend(fontsize='small', loc='best')

fig.tight_layout()
plt.show()


In [ ]:
# SurvSHAP(t) quantitative recovery — Dataset A (Sign Recovery, Rank Agreement, Precision@k)
# b_hat_i = mean of phi_i(t) over interp_times (time-averaged SHAP value per individual)
from scipy.stats import spearmanr

N_SHAP_A  = 10
rng_shap_a = np.random.default_rng(SEED)
shap_a_idx = rng_shap_a.choice(len(X_int_te), size=N_SHAP_A, replace=False)

imp_idx_a = [feature_names.index(n) for n in IMPORTANT_FEATURES]
k_imp_a   = len(imp_idx_a)
beta_S_a  = TRUE_BETAS_A[imp_idx_a]

shap_a_summary = []
for mname in MODELS_A:
    expl = SurvSHAPExplainer(
        interp_models[mname], X_int_tr, T_int_tr, E_int_tr,
        feature_names=feature_names, function_type='chf',
    )
    sign_rec_per = []
    rank_agr_per = []
    prec_k_per   = []

    for i in shap_a_idx:
        result = expl.explain(X_int_te[i], interp_times)
        _, var_names, shap_mat = SurvSHAPExplainer._extract_shap_matrix(result)
        order    = [list(var_names).index(n) for n in feature_names]
        shap_mat = shap_mat[order]          # (p, n_t)
        b_hat    = shap_mat.mean(axis=1)    # time-averaged → (p,)
        b_hat_S  = b_hat[imp_idx_a]

        sign_rec_per.append(float(np.mean(np.sign(b_hat_S) == np.sign(beta_S_a))))

        rho, _ = spearmanr(np.abs(beta_S_a), np.abs(b_hat_S))
        if not np.isnan(rho):
            rank_agr_per.append(rho)

        top_k_set = set(np.argsort(np.abs(b_hat))[::-1][:k_imp_a])
        prec_k_per.append(len(top_k_set & set(imp_idx_a)) / k_imp_a)

    shap_a_summary.append({
        'model':          mname,
        'N':              N_SHAP_A,
        'sign_recovery':  f'{np.mean(sign_rec_per):.3f}',
        'rank_agreement': f'{np.mean(rank_agr_per):.3f}',
        'precision@k':    f'{np.mean(prec_k_per):.3f}',
    })

shap_a_recovery_df = pd.DataFrame(shap_a_summary)
print(f'=== SurvSHAP(t) quantitative recovery (Dataset A, N={N_SHAP_A}, k={k_imp_a}) ===')
print(f'True betas (informative, constant): {beta_S_a}')
print()
print(shap_a_recovery_df.to_string(index=False))
shap_a_recovery_df

## 4. Evaluation on Dataset B

Stress test for SurvSHAP(t): time-varying $\beta_1(t) = -1 + 2t$ with a sign change at $t = 0.5$.


### 4.1 SurvLIME

SurvLIME's main limitation: one coefficient per feature cannot capture that $\beta_1(t) = -1 + 2t$ grows and changes sign at $t = 0.5$. SurvLIME will produce one averaged coefficient for $x_1$.

Expected: SurvLIME can find important features but collapses the time-varying effect to a single number → temporal dynamics are lost.


### 4.2 SurvSHAP(t)

SurvSHAP(t)'s main advantage is temporal feature dynamics. For quantitative metrics: $\hat{b}_i = \frac{1}{|\mathcal{T}|}\sum_{t \in \mathcal{T}}\phi_i(t)$ (time-averaged), compared against time-averaged true $\beta_i(t)$ (`TRUE_BETAS_B_AVG`).

- **Sign Recovery**: fraction of informative features with the correct sign of the time-averaged effect.
- **Rank Agreement**: $\rho_s(|\beta_S|, |\hat{b}_S|)$ — correct ranking of informative features.
- **Precision@k**: fraction of top-4 features by $|\hat{b}|$ that belong to $S$.

For CoxPH (constant log-hazard ratios) curves are expected nearly flat regardless of true dynamics; non-parametric RSF should better capture non-monotone attribution → expected higher Sign Recovery and Rank Agreement.

In [ ]:
# SurvLIME quantitative recovery — Dataset B (Sign Recovery, Rank Agreement, Precision@k)
# Reference: time-averaged ground-truth betas over the evaluation time grid
from scipy.stats import spearmanr

TRUE_BETAS_B_AVG = np.array([
    1.0,
    float((-1.0 + 2.0 * tv_times).mean()),  # beta1(t) = -1+2t, time-averaged over tv_times
    -0.2,
    0.1,
    1e-6, 1e-6, 1e-6, 1e-6,
])
print('Ground-truth betas for Dataset B, time-averaged over evaluation grid:')
for name, b in zip(feature_names, TRUE_BETAS_B_AVG):
    marker = '*' if name in IMPORTANT_FEATURES else ' '
    print(f'  {marker} {name}: {b:+.4f}')
print()

imp_idx_b = [feature_names.index(n) for n in IMPORTANT_FEATURES]
k_imp_b   = len(imp_idx_b)
beta_S_b  = TRUE_BETAS_B_AVG[imp_idx_b]

N_LIME_B = 10
rng_lime_b = np.random.default_rng(SEED)
lime_b_idx = rng_lime_b.choice(len(X_tv_te), size=N_LIME_B, replace=False)

lime_b_summary = []
for mname in ['cox_ph', 'rsf']:
    lime_expl = SurvLIMEExplainer(tv_models[mname], X_tv_tr, T_tv_tr, E_tv_tr)
    sign_rec_per = []
    rank_agr_per = []
    prec_k_per   = []

    for i in lime_b_idx:
        coefs   = lime_expl.explain(X_tv_te[i], num_samples=500)
        b_hat_S = coefs[imp_idx_b]

        sign_rec_per.append(float(np.mean(np.sign(b_hat_S) == np.sign(beta_S_b))))

        rho, _ = spearmanr(np.abs(beta_S_b), np.abs(b_hat_S))
        if not np.isnan(rho):
            rank_agr_per.append(rho)

        top_k_set = set(np.argsort(np.abs(coefs))[::-1][:k_imp_b])
        prec_k_per.append(len(top_k_set & set(imp_idx_b)) / k_imp_b)

    lime_b_summary.append({
        'model':          mname,
        'N':              N_LIME_B,
        'sign_recovery':  f'{np.mean(sign_rec_per):.3f}',
        'rank_agreement': f'{np.mean(rank_agr_per):.3f}',
        'precision@k':    f'{np.mean(prec_k_per):.3f}',
    })

lime_b_recovery_df = pd.DataFrame(lime_b_summary)
print(f'=== SurvLIME quantitative recovery (Dataset B, N={N_LIME_B}, k={k_imp_b}) ===')
print(f'Reference betas (informative, time-avg): {beta_S_b}')
print()
print(lime_b_recovery_df.to_string(index=False))
lime_b_recovery_df

### 4.2 SurvSHAP(t)

Here is SurvSHAP(t)'s main advantage. Evaluations:

1. **SurvSHAP(t) curves** for $x_1$ — the curve should change sign near $t = 0.5$ (V-shape for $x_1 = 1$).
2. **Changing Sign Proportion (CSP)** — higher for $x_1$ than for constant and noise features.
3. **Time-wise rank correlation** between $|\phi_j(t)|$ and $|\beta_j(t)\,x_j|$, averaged over time.
4. **Noise stability**: $x_4..x_7$ should have small $|\phi|$.

For CoxPH (constant log-hazard ratios), curves are expected nearly flat regardless of true dynamics; non-parametric RSF should capture non-monotone attribution.


In [ ]:
# 5.2b SurvSHAP(t) — Dataset B (Non-PH, time-varying x1): CPH vs RSF
risk_tv = tv_models['rsf'].predict_risk(X_tv_te)

# x1 ~ Bernoulli(0.5):
#   beta1(t)*x1 = (-1+2t)*x1   contribution V-shape (min at t=0.5, zero at t=1) for x1=1
#                              identically zero for x1=0
# x3 has constant beta3=0.1, so its SurvSHAP(t) curve should be ~flat.
idx_b = int(np.argmin(np.abs(risk_tv - np.percentile(risk_tv, 50))))

print(f'Dataset B - selected individual: index={idx_b}, risk={risk_tv[idx_b]:.3f}')
print(f'Observed time: {T_tv_te[idx_b]:.3f}, event: {int(E_tv_te[idx_b])}')
print()
print('Feature values:')
for name, val in zip(feature_names, X_tv_te[idx_b]):
    marker = ' *' if name in IMPORTANT_FEATURES else ''
    print(f'  {name}: {val:+.3f}{marker}')

shap_results_b = {}
for mname in ['cox_ph', 'rsf']:
    expl = SurvSHAPExplainer(
        tv_models[mname], X_tv_tr, T_tv_tr, E_tv_tr,
        feature_names=feature_names,
        function_type="chf",
    )
    shap_results_b[mname] = expl.explain(X_tv_te[idx_b], tv_times)
    print(f'  {mname}: done')

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
fig.suptitle(
    'SurvSHAP(t) - Dataset B (Non-PH, time-varying x1)\n'
    'Target: H(t).  beta1(t) = -1 + 2t  (sign change t = 0.5)',
    fontsize=12,
)

for col, mname in enumerate(['cox_ph', 'rsf']):
    result_df = shap_results_b[mname]
    times_b, var_names_b, shap_mat = SurvSHAPExplainer._extract_shap_matrix(result_df)
    norm_mat = SurvSHAPExplainer._normalize_shap(shap_mat)

    for row, (matrix, ylabel) in enumerate(zip(
        [shap_mat, norm_mat],
        ['SurvSHAP(t) for H(t)', 'Normalized SurvSHAP(t)'],
    )):
        ax = axes[row, col]
        for i, name in enumerate(var_names_b):
            if name in IMPORTANT_FEATURES:
                ax.plot(times_b, matrix[i], label=name, linewidth=2, color=feat_color[name])
            else:
                ax.plot(times_b, matrix[i], alpha=0.2, color='gray', linewidth=0.8)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0.5, color='red', linestyle=':', linewidth=1, alpha=0.6,
                   label='_nolegend_' if row else 'true sign change (t=0.5)')
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(f'{mname.upper().replace("_", " ")} model')
        if row == 1:
            ax.set_xlabel('Time')
        if col == 1 and row == 0:
            ax.legend(fontsize='small', loc='best')

fig.tight_layout()
plt.show()


In [ ]:
# Global SurvSHAP importance - Dataset B (TV), CoxPH vs RSF
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, mname in zip(axes, ['cox_ph', 'rsf']):
    shap_expl = SurvSHAPExplainer(
        tv_models[mname], X_tv_tr, T_tv_tr, E_tv_tr,
        feature_names=feature_names,
        function_type="chf",
    )
    importance_df = shap_expl.explain_multiple(X_tv_te[:30], tv_times)
    shap_expl.plot_global_importance(importance_df, feature_names, ax=ax,
                                     highlight_features=IMPORTANT_FEATURES)
    ax.set_title(f'{mname.upper()} — Global SurvSHAP Importance (Dataset B)')

fig.suptitle('Global Feature Importance on Non-PH Data [target: H(t)]\n(x0–x3 important; noise x4–x7 should rank low)',
             fontsize=12, y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
# SurvSHAP(t) quantitative recovery — Dataset B (Sign Recovery, Rank Agreement, Precision@k)
# b_hat_i = mean of phi_i(t) over tv_times; reference = TRUE_BETAS_B_AVG (time-averaged)
from scipy.stats import spearmanr

N_EXPL = 30
expl_idx = np.arange(min(N_EXPL, X_tv_te.shape[0]))

imp_idx_shap_b = [feature_names.index(n) for n in IMPORTANT_FEATURES]
k_shap_b       = len(imp_idx_shap_b)
beta_S_shap_b  = TRUE_BETAS_B_AVG[imp_idx_shap_b]

shap_b_summary = []
for mname in ['cox_ph', 'rsf']:
    expl = SurvSHAPExplainer(
        tv_models[mname], X_tv_tr, T_tv_tr, E_tv_tr,
        feature_names=feature_names, function_type='chf',
    )
    sign_rec_per = []
    rank_agr_per = []
    prec_k_per   = []

    for i in expl_idx:
        result = expl.explain(X_tv_te[i], tv_times)
        _, var_names, shap_mat = SurvSHAPExplainer._extract_shap_matrix(result)
        order    = [list(var_names).index(n) for n in feature_names]
        shap_mat = shap_mat[order]          # (p, n_t)
        b_hat    = shap_mat.mean(axis=1)    # time-averaged → (p,)
        b_hat_S  = b_hat[imp_idx_shap_b]

        sign_rec_per.append(float(np.mean(np.sign(b_hat_S) == np.sign(beta_S_shap_b))))

        rho, _ = spearmanr(np.abs(beta_S_shap_b), np.abs(b_hat_S))
        if not np.isnan(rho):
            rank_agr_per.append(rho)

        top_k_set = set(np.argsort(np.abs(b_hat))[::-1][:k_shap_b])
        prec_k_per.append(len(top_k_set & set(imp_idx_shap_b)) / k_shap_b)

    shap_b_summary.append({
        'model':          mname,
        'N':              len(expl_idx),
        'sign_recovery':  f'{np.mean(sign_rec_per):.3f}',
        'rank_agreement': f'{np.mean(rank_agr_per):.3f}',
        'precision@k':    f'{np.mean(prec_k_per):.3f}',
    })

shap_b_recovery_df = pd.DataFrame(shap_b_summary)
print(f'=== SurvSHAP(t) quantitative recovery (Dataset B, N={len(expl_idx)}, k={k_shap_b}) ===')
print(f'Reference betas (informative, time-avg): {beta_S_shap_b}')
print()
print(shap_b_recovery_df.to_string(index=False))
shap_b_recovery_df

## 5. Summary comparison

Final table: SurvLIME vs SurvSHAP(t) across both settings.


In [ ]:
# Final summary: Sign Recovery / Rank Agreement / Precision@k for all combinations.
rows = []
for df, dataset, method in [
    (lime_recovery_df,   'A (PH, constant)',  'SurvLIME'),
    (shap_a_recovery_df, 'A (PH, constant)',  'SurvSHAP(t)'),
    (lime_b_recovery_df, 'B (Non-PH, TV x1)', 'SurvLIME'),
    (shap_b_recovery_df, 'B (Non-PH, TV x1)', 'SurvSHAP(t)'),
]:
    for _, row in df.iterrows():
        rows.append({
            'dataset':        dataset,
            'method':         method,
            'model':          row['model'],
            'sign_recovery':  row['sign_recovery'],
            'rank_agreement': row['rank_agreement'],
            'precision@k':    row['precision@k'],
        })

summary_df = pd.DataFrame(rows)
print('=== Evaluation Summary: Sign Recovery / Rank Agreement / Precision@k ===')
print(summary_df.to_string(index=False))
summary_df

## 6. Conclusions

**Dataset A** — SurvLIME works well for recovering constant Cox-like effects: it correctly identifies informative variables, their signs, and approximate ranking.

**Dataset B** — SurvLIME's main limitation: one coefficient per feature cannot represent time-varying effects. SurvSHAP(t) is harder to evaluate numerically, but Dataset B reveals its strength — it captures the temporal dynamics of feature effects.

| Method             | Strong side                                                           | Weak side                                    |
| ------------------ | --------------------------------------------------------------------- | -------------------------------------------- |
| SurvLIME           | simple coefficient interpretation, good for Cox-like constant effects | cannot represent time-varying effects        |
| SurvSHAP(t)        | explains feature effects over time                                    | harder to evaluate and interpret numerically |
| SurvLIME on RSF    | still finds important variables                                       | less stable coefficients                     |
| SurvSHAP(t) on RSF | can show temporal behavior                                            | curves may be noisy                          |
